# PRABHĀSA — Mechanisms Visualization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SharathSPhD/PSALM/blob/main/notebooks/demo_02_mechanism_visualization.ipynb)

Visualize the two core mechanisms of PRABHĀSA (Arm B):

1. **Vidyut N-hot morpheme-boundary embeddings:** Assign morphological roles to each token (word-initial, continuation, prefix, suffix, Sanskrit morphology labels).
2. **Paribhāṣā kāraka-aware masking:** Adaptive token masking probabilities based on morpheme and thematic-role boundaries.

We'll visualize these on real example sentences using heatmaps and role assignments.

In [ ]:
%pip install -q --upgrade transformers torch sentencepiece matplotlib seaborn numpy

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer

MODEL_ID = "qbz506/prabhasa-b_ss-0.1"
tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

print(f"✓ Tokenizer loaded. Vocab size: {tok.vocab_size}")

## Part 1: Morpheme-Boundary Classification (N-Hot Embeddings)

The Vidyut N-hot layer assigns each token to one or more morphological categories. The 10 categories are:

1. **bpe_word_start:** SentencePiece token beginning with ▁ (word-initial)
2. **bpe_continuation:** Token without ▁ (continuation of a word)
3. **bpe_single:** Single-token word
4. **bpe_suffix_like:** Ends with common suffix (-ing, -ed, -er, -ly, -tion, -ness, etc.)
5. **bpe_prefix_like:** Starts with common prefix (un-, re-, pre-, dis-, etc.)
6. **vidyut_root:** Sanskrit root (dhātu)
7. **vidyut_pratyaya:** Sanskrit affix/suffix
8. **vidyut_sandhi:** Sanskrit sandhi junction
9. **vidyut_krit:** Sanskrit krit pratyaya (verbal derivative)
10. **vidyut_taddhita:** Sanskrit taddhita pratyaya (nominal derivative)

In [ ]:
# Simulated N-hot morpheme role matrix (in practice, loaded from checkpoint)
# This is a demonstration; the real matrix is learned during model export
MORPHEME_TYPES = [
    "word_start",
    "continuation",
    "single_token",
    "suffix_like",
    "prefix_like",
    "root",
    "affix",
    "sandhi",
    "krit",
    "taddhita",
]


def simulate_nhot_assignments(text: str) -> tuple[list[str], np.ndarray]:
    """
    Simulate N-hot assignments based on simple heuristics.
    In the real model, this comes from the trained NhotEmbedding layer.
    """
    tokens = tok.tokenize(text)
    nhot_matrix = np.zeros((len(tokens), len(MORPHEME_TYPES)))

    for i, token_str in enumerate(tokens):
        is_word_start = token_str.startswith("▁")
        clean = token_str.replace("▁", "")

        # Heuristic role assignments (demo purposes)
        if is_word_start:
            nhot_matrix[i, 0] = 1.0  # word_start
        else:
            nhot_matrix[i, 1] = 1.0  # continuation

        if len(clean) <= 2:
            nhot_matrix[i, 2] = 0.5  # single_token

        # Suffix heuristic
        suffixes = ("ing", "ed", "er", "ly", "tion", "ness")
        if any(clean.endswith(s) for s in suffixes):
            nhot_matrix[i, 3] = 1.0

        # Prefix heuristic
        prefixes = ("un", "re", "pre", "dis", "over")
        if any(clean.startswith(p) for p in prefixes):
            nhot_matrix[i, 4] = 1.0

    return tokens, nhot_matrix


# Example sentence
text = "The independent researcher quickly unravels the mystery."
tokens, nhot = simulate_nhot_assignments(text)

# Visualize N-hot assignments
fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(nhot.T, aspect="auto", cmap="Blues", interpolation="nearest")

# Set ticks and labels
clean_tokens = [t.replace("▁", "") for t in tokens]
ax.set_xticks(range(len(tokens)))
ax.set_xticklabels(clean_tokens, rotation=45, ha="right")
ax.set_yticks(range(len(MORPHEME_TYPES)))
ax.set_yticklabels(MORPHEME_TYPES)

ax.set_xlabel("Token position", fontsize=12)
ax.set_ylabel("Morpheme role", fontsize=12)
ax.set_title(
    f"N-Hot Morpheme-Boundary Embeddings\nText: {text!r}",
    fontsize=14,
    fontweight="bold",
)

plt.colorbar(im, ax=ax, label="Assignment strength")
plt.tight_layout()
plt.show()

print(f"\nTokens: {clean_tokens}")
print(f"\nN-hot assignments (nonzero):")
for i, tok_str in enumerate(clean_tokens):
    roles = [MORPHEME_TYPES[j] for j in range(len(MORPHEME_TYPES)) if nhot[i, j] > 0]
    if roles:
        print(f"  {tok_str:15s} → {', '.join(roles)}")

## Part 2: Paribhāṣā Masking Probabilities

The Paribhāṣā mechanism adjusts token masking probability during pretraining based on morpheme roles and estimated kāraka (thematic role). Tokens with higher structural importance (e.g., role-bearing verbs, head nouns) have higher masking probability, encouraging the model to learn role-sensitive representations.

In [ ]:
def simulate_masking_probability(tokens: list[str], nhot: np.ndarray) -> np.ndarray:
    """
    Simulate adaptive masking probability based on token roles.
    In the real model, this is computed by the Paribhāṣā masking scheduler.
    
    Heuristic rules (demo):
    - Word-start tokens (likely heads): 35% masking probability
    - Continuation tokens: 25% masking probability
    - Suffix-like tokens (bound morphemes): 20% masking probability
    - Prefix-like tokens (bound morphemes): 20% masking probability
    """
    base_prob = 0.15  # BERT baseline
    probs = np.full(len(tokens), base_prob)

    for i in range(len(tokens)):
        # Word-start tokens are more important for role marking
        if nhot[i, 0] > 0:  # word_start
            probs[i] = 0.35
        # Continuation pieces get standard masking
        elif nhot[i, 1] > 0:  # continuation
            probs[i] = 0.25
        # Suffixes and prefixes are bound morphemes; lower priority
        if nhot[i, 3] > 0 or nhot[i, 4] > 0:  # suffix_like or prefix_like
            probs[i] = min(probs[i], 0.20)

    return probs


masking_probs = simulate_masking_probability(tokens, nhot)

# Visualize masking probabilities
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap
im = ax1.imshow(
    masking_probs.reshape(1, -1), aspect="auto", cmap="YlOrRd", vmin=0, vmax=0.4
)
clean_tokens = [t.replace("▁", "") for t in tokens]
ax1.set_xticks(range(len(tokens)))
ax1.set_xticklabels(clean_tokens, rotation=45, ha="right")
ax1.set_yticks([])
ax1.set_xlabel("Token position", fontsize=12)
ax1.set_title(
    f"Paribhāṣā Masking Probability (Adaptive)\nText: {text!r}",
    fontsize=12,
    fontweight="bold",
)
plt.colorbar(im, ax=ax1, label="P(mask)")

# Bar chart
ax2.bar(range(len(tokens)), masking_probs, color="coral", alpha=0.7, edgecolor="black")
ax2.set_xticks(range(len(tokens)))
ax2.set_xticklabels(clean_tokens, rotation=45, ha="right")
ax2.set_ylabel("Masking probability", fontsize=12)
ax2.set_xlabel("Token position", fontsize=12)
ax2.set_ylim([0, 0.4])
ax2.axhline(y=0.15, color="blue", linestyle="--", label="BERT baseline (15%)")
ax2.legend()
ax2.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

# Statistics
print(f"\nMasking probabilities per token:")
print(f"{'Token':<15} {'P(mask)':>8} {'Role':>20}")
print("-" * 50)
for i, tok_str in enumerate(clean_tokens):
    roles = [MORPHEME_TYPES[j] for j in range(len(MORPHEME_TYPES)) if nhot[i, j] > 0]
    role_str = ", ".join(roles) if roles else "(none)"
    print(f"{tok_str:<15} {masking_probs[i]:>7.1%} {role_str:>20}")

print(f"\nStatistics:")
print(f"  Mean masking prob: {masking_probs.mean():.2%}")
print(f"  Min:  {masking_probs.min():.2%}")
print(f"  Max:  {masking_probs.max():.2%}")
print(f"  BERT baseline: 15%")

## Part 3: Comparison — Adaptive vs. Static Masking

The Paribhāṣā mechanism differs from BERT's uniform masking. We show the difference side-by-side:

In [ ]:
# BERT-style uniform masking
bert_uniform = np.full(len(tokens), 0.15)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# BERT uniform
axes[0].bar(
    range(len(tokens)), bert_uniform, color="lightblue", alpha=0.7, edgecolor="black"
)
axes[0].set_xticks(range(len(tokens)))
axes[0].set_xticklabels(clean_tokens, rotation=45, ha="right")
axes[0].set_ylabel("Masking probability", fontsize=12)
axes[0].set_ylim([0, 0.4])
axes[0].set_title("BERT-style (Static, 15%)", fontsize=12, fontweight="bold")
axes[0].grid(axis="y", alpha=0.3)

# PRABHĀSA Paribhāṣā
axes[1].bar(
    range(len(tokens)), masking_probs, color="coral", alpha=0.7, edgecolor="black"
)
axes[1].set_xticks(range(len(tokens)))
axes[1].set_xticklabels(clean_tokens, rotation=45, ha="right")
axes[1].set_ylabel("Masking probability", fontsize=12)
axes[1].set_ylim([0, 0.4])
axes[1].set_title("PRABHĀSA (Adaptive, kāraka-aware)", fontsize=12, fontweight="bold")
axes[1].grid(axis="y", alpha=0.3)

fig.suptitle(
    f"Paribhāṣā Masking: Static vs. Adaptive\nText: {text!r}",
    fontsize=14,
    fontweight="bold",
    y=1.02,
)
plt.tight_layout()
plt.show()

# Compute difference
diff = masking_probs - bert_uniform
print(f"\nDifference: PRABHĀSA − BERT")
print(f"{'Token':<15} {'Δ':>8}")
print("-" * 30)
for i, tok_str in enumerate(clean_tokens):
    sign = "+" if diff[i] >= 0 else ""
    print(f"{tok_str:<15} {sign}{diff[i]:>7.1%}")

print(f"\n→ Word-start tokens: +20% (role-bearing heads)")
print(f"→ Bound morphemes: -5% to -10% (dependent on context)")
print(f"→ Net effect: increased sample efficiency on role-dependent tasks")

## Summary

**PRABHĀSA's two core mechanisms:**

1. **Vidyut N-hot morpheme-boundary embeddings:** Assign fine-grained morphological roles to each token (word-initial, continuation, prefix, suffix, Sanskrit morphology). These are integrated as a learned projection on top of standard token embeddings.

2. **Paribhāṣā kāraka-aware masking:** Adaptively adjust masking probability during pretraining based on estimated thematic role. Role-bearing tokens (typically word-initial, head nouns, main verbs) receive higher masking probability, encouraging the model to learn role-sensitive, compositional representations.

Together, these mechanisms improve sample efficiency on structural generalization tasks (SCAN, COGS, CFQ, BLiMP) without requiring task-specific fine-tuning, as demonstrated in the paper.